In [26]:
# Install required packages
!pip install nltk python-Levenshtein matplotlib torch


In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
import json
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import nltk
import Levenshtein
from collections import Counter
import math
import random

# Download required NLTK data

nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)


True

In [28]:
def encode_sentence(sentence, token2id, is_urdu=True):
    """Encode a sentence using greedy longest-match subword tokenization."""
    tokens = []

    if is_urdu:
        words = ["_" + w for w in sentence.split()]
    else:
        words = [w + "_" for w in sentence.split()]

    for w in words:
        i = 0
        while i < len(w):
            subword = None
            for j in range(len(w), i, -1):
                piece = w[i:j]
                if piece in token2id:
                    subword = piece
                    break
            if subword is None:
                tokens.append(token2id["<unk>"])
                i += 1
            else:
                tokens.append(token2id[subword])
                i += len(subword)

    return [token2id["<sos>"]] + tokens + [token2id["<eos>"]]



class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab):
        self.src_data = []
        self.tgt_data = []

        for src, tgt in zip(src_sentences, tgt_sentences):
            src_tokens = encode_sentence(src, src_vocab, is_urdu=True)
            tgt_tokens = encode_sentence(tgt, tgt_vocab, is_urdu=False)

            self.src_data.append(torch.tensor(src_tokens))
            self.tgt_data.append(torch.tensor(tgt_tokens))

    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, idx):
        return self.src_data[idx], self.tgt_data[idx]


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)

    # lengths
    max_src_len = max(len(src) for src in src_batch)
    max_tgt_len = max(len(tgt) for tgt in tgt_batch)

    # we force both to same max length
    max_len = max(max_src_len, max_tgt_len)

    # pad each sequence with 0 up to max_len
    src_padded = [F.pad(src, (0, max_len - len(src)), value=0) for src in src_batch]
    tgt_padded = [F.pad(tgt, (0, max_len - len(tgt)), value=0) for tgt in tgt_batch]

    # stack into batch tensors
    return torch.stack(src_padded), torch.stack(tgt_padded)



In [29]:


class Encoder(nn.Module):
    def __init__(self, vocab_size=512, embed_dim=128, hidden_dim=128, num_layers=2, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.bilstm = nn.LSTM(embed_dim, hidden_dim, num_layers,
                             dropout=dropout, bidirectional=True, batch_first=True)
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

    def forward(self, x):
        embedded = self.embedding(x)
        embedded = self.dropout(embedded)
        output, (h, c) = self.bilstm(embedded)
        return output, (h, c)

class Decoder(nn.Module):
    def __init__(self, vocab_size=512, embed_dim=256, hidden_dim=256, num_layers=4,
                 output_vocab_size=512, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers,
                           dropout=dropout, batch_first=True)
        self.linear = nn.Linear(hidden_dim, output_vocab_size)
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        # Project encoder states to decoder dimensions
        self.h_projection = nn.Linear(256, hidden_dim)  # 256 from bidirectional encoder
        self.c_projection = nn.Linear(256, hidden_dim)

    def forward(self, x, encoder_states=None):
        embedded = self.embedding(x)
        embedded = self.dropout(embedded)

        if encoder_states is not None:
            h_enc, c_enc = encoder_states
            # Convert bidirectional encoder states to decoder format
            # h_enc: [num_layers*2, batch, hidden_dim] -> [num_layers, batch, hidden_dim*2]
            batch_size = h_enc.size(1)
            h_enc = h_enc.view(2, 2, batch_size, -1)  # [directions, layers, batch, hidden]
            c_enc = c_enc.view(2, 2, batch_size, -1)

            # Concatenate forward and backward states
            h_enc = torch.cat([h_enc[0], h_enc[1]], dim=-1)  # [layers, batch, hidden*2]
            c_enc = torch.cat([c_enc[0], c_enc[1]], dim=-1)

            # Project to decoder dimensions and repeat for all decoder layers
            h_init = self.h_projection(h_enc[-1]).unsqueeze(0).repeat(self.num_layers, 1, 1)
            c_init = self.c_projection(c_enc[-1]).unsqueeze(0).repeat(self.num_layers, 1, 1)

            initial_state = (h_init, c_init)
        else:
            initial_state = None

        output, _ = self.lstm(embedded, initial_state)
        output = self.linear(output)
        return output

class Seq2SeqModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, src):
        # Encoder processes source
        enc_output, enc_states = self.encoder(src)

        # Decoder processes same source sequence (as per your requirement)
        dec_output = self.decoder(src, enc_states)

        return dec_output

In [30]:
def load_data_and_vocab():
    """Load data and vocabularies from Google Drive"""
    base_path = "/content/drive/MyDrive/Model2/"

    # Load source and target sentences
    with open(base_path + "src_normalized.txt", 'r', encoding='utf-8') as f:
        src_sentences = [line.strip() for line in f]

    with open(base_path + "tgt_normalized.txt", 'r', encoding='utf-8') as f:
        tgt_sentences = [line.strip() for line in f]

    # Load vocabularies
    with open(base_path + "vocab_Urdu.json", 'r', encoding='utf-8') as f:
        urdu_vocab = json.load(f)

    with open(base_path + "vocab_Roman.json", 'r', encoding='utf-8') as f:
        roman_vocab = json.load(f)

    return src_sentences, tgt_sentences, urdu_vocab, roman_vocab

def create_datasets(src_sentences, tgt_sentences, urdu_vocab, roman_vocab):
    """Create train/val/test splits"""
    total_size = len(src_sentences)
    train_size = int(0.7 * total_size)
    val_size = int(0.15 * total_size)

    # Shuffle data
    indices = list(range(total_size))
    random.shuffle(indices)

    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:]

    # Create datasets
    train_src = [src_sentences[i] for i in train_indices]
    train_tgt = [tgt_sentences[i] for i in train_indices]

    val_src = [src_sentences[i] for i in val_indices]
    val_tgt = [tgt_sentences[i] for i in val_indices]

    test_src = [src_sentences[i] for i in test_indices]
    test_tgt = [tgt_sentences[i] for i in test_indices]

    train_dataset = TranslationDataset(train_src, train_tgt, urdu_vocab, roman_vocab)
    val_dataset = TranslationDataset(val_src, val_tgt, urdu_vocab, roman_vocab)
    test_dataset = TranslationDataset(test_src, test_tgt, urdu_vocab, roman_vocab)

    return train_dataset, val_dataset, test_dataset

def calculate_perplexity(loss):
    """Calculate perplexity from loss"""
    return math.exp(loss)

def decode_tokens(tokens, id2token):
    """Convert token IDs back to text"""
    words = []
    for token_id in tokens:
        if token_id in [0, 1, 2]:  # pad, sos, eos
            continue
        words.append(id2token.get(token_id, '<unk>'))
    return ' '.join(words)

def calculate_bleu(reference, hypothesis):
    """Calculate BLEU score"""
    reference_tokens = reference.split()
    hypothesis_tokens = hypothesis.split()

    if len(hypothesis_tokens) == 0:
        return 0.0

    smoothie = SmoothingFunction().method4
    return sentence_bleu([reference_tokens], hypothesis_tokens, smoothing_function=smoothie)

def calculate_cer(reference, hypothesis):
    """Calculate Character Error Rate"""
    if len(reference) == 0:
        return 1.0 if len(hypothesis) > 0 else 0.0
    return Levenshtein.distance(reference, hypothesis) / len(reference)

def calculate_edit_distance(reference, hypothesis):
    """Calculate Levenshtein distance"""
    return Levenshtein.distance(reference, hypothesis)


In [31]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, device, roman_vocab):
    id2roman = {v: k for k, v in roman_vocab.items()}

    for epoch in range(epochs):
        # ---- TRAINING ----
        model.train()
        total_loss = 0
        correct, total = 0, 0

        for batch_idx, (src, tgt) in enumerate(train_loader):
            src, tgt = src.to(device), tgt.to(device)

            optimizer.zero_grad()
            output = model(src)  # (batch, seq_len, vocab_size)

            # Flatten for CE Loss
            output_flat = output.reshape(-1, output.size(-1))
            tgt_flat = tgt.reshape(-1)

            loss = criterion(output_flat, tgt_flat)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()

            # Accuracy (token-level)
            predictions = torch.argmax(output, dim=-1)
            correct += (predictions == tgt).sum().item()
            total += tgt.numel()

        avg_train_loss = total_loss / len(train_loader)
        train_perplexity = calculate_perplexity(avg_train_loss)
        train_accuracy = correct / total

        # ---- VALIDATION ----
        model.eval()
        val_loss, val_bleu, val_cer, val_edit, val_correct, val_total = 0, 0, 0, 0, 0, 0

        with torch.no_grad():
            for src, tgt in val_loader:
                src, tgt = src.to(device), tgt.to(device)
                output = model(src)

                # Loss
                output_flat = output.reshape(-1, output.size(-1))
                tgt_flat = tgt.reshape(-1)
                loss = criterion(output_flat, tgt_flat)
                val_loss += loss.item()

                # Predictions
                predictions = torch.argmax(output, dim=-1)
                val_correct += (predictions == tgt).sum().item()
                val_total += tgt.numel()

                for i in range(src.size(0)):
                    pred_tokens = predictions[i].cpu().numpy()
                    tgt_tokens = tgt[i].cpu().numpy()

                    pred_text = decode_tokens(pred_tokens, id2roman)
                    ref_text = decode_tokens(tgt_tokens, id2roman)

                    val_bleu += calculate_bleu(ref_text, pred_text)
                    val_cer += calculate_cer(ref_text, pred_text)
                    val_edit += calculate_edit_distance(ref_text, pred_text)

        # Averages
        avg_val_loss = val_loss / len(val_loader)
        val_perplexity = calculate_perplexity(avg_val_loss)
        avg_val_bleu = val_bleu / len(val_loader.dataset)
        avg_val_cer = val_cer / len(val_loader.dataset)
        avg_val_edit = val_edit / len(val_loader.dataset)
        val_accuracy = val_correct / val_total

        # ---- RESULTS ----
        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"Train Loss: {avg_train_loss:.4f}, Perplexity: {train_perplexity:.4f}, Accuracy: {train_accuracy:.4f}")
        print(f"Val Loss:   {avg_val_loss:.4f}, Perplexity: {val_perplexity:.4f}, "
              f"Accuracy: {val_accuracy:.4f}, BLEU: {avg_val_bleu:.4f}, CER: {avg_val_cer:.4f}, "
              f"Edit Dist: {avg_val_edit:.2f}")




def evaluate_model(model, test_loader, roman_vocab, device):
    """Evaluate model with metrics"""
    model.eval()
    id2roman = {v: k for k, v in roman_vocab.items()}

    total_bleu = 0
    total_cer = 0
    total_edit_dist = 0
    total_loss = 0
    count = 0

    criterion = nn.CrossEntropyLoss(ignore_index=0)

    with torch.no_grad():
        for src, tgt in test_loader:
            src, tgt = src.to(device), tgt.to(device)

            output = model(src)

            # Calculate loss
            loss_output_flat = output.reshape(-1, output.size(-1))
            target_flat = tgt.reshape(-1)
            loss = criterion(loss_output_flat, target_flat)
            total_loss += loss.item()

            predictions = torch.argmax(output, dim=-1)

            for i in range(src.size(0)):
                pred_tokens = predictions[i].cpu().numpy()
                tgt_tokens = tgt[i].cpu().numpy()

                pred_text = decode_tokens(pred_tokens, id2roman)
                ref_text = decode_tokens(tgt_tokens, id2roman)

                total_bleu += calculate_bleu(ref_text, pred_text)
                total_cer += calculate_cer(ref_text, pred_text)
                total_edit_dist += calculate_edit_distance(ref_text, pred_text)
                count += 1

    avg_loss = total_loss / len(test_loader)
    avg_bleu = total_bleu / count
    avg_cer = total_cer / count
    avg_edit_dist = total_edit_dist / count

    print(f"Test Loss: {avg_loss:.4f}, Perplexity: {calculate_perplexity(avg_loss):.4f}")
    print(f"BLEU Score: {avg_bleu:.4f}")
    print(f"Character Error Rate: {avg_cer:.4f}")
    print(f"Average Edit Distance: {avg_edit_dist:.2f}")

    return avg_bleu, avg_cer, avg_edit_dist

def show_examples(model, test_dataset, urdu_vocab, roman_vocab, device, num_examples=5):
    """Show translation examples"""
    model.eval()
    id2roman = {v: k for k, v in roman_vocab.items()}
    id2urdu = {v: k for k, v in urdu_vocab.items()}

    indices = random.sample(range(len(test_dataset)), num_examples)

    with torch.no_grad():
        for idx in indices:
            src, tgt = test_dataset[idx]
            src_batch = src.unsqueeze(0).to(device)

            output = model(src_batch)
            prediction = torch.argmax(output, dim=-1).squeeze(0)

            src_text = decode_tokens(src.numpy(), id2urdu)
            tgt_text = decode_tokens(tgt.numpy(), id2roman)
            pred_text = decode_tokens(prediction.cpu().numpy(), id2roman)

            print(f"Source (Urdu): {src_text}")
            print(f"Target (Roman): {tgt_text}")
            print(f"Prediction: {pred_text}")

            # Calculate metrics for this example
            bleu = calculate_bleu(tgt_text, pred_text)
            cer = calculate_cer(tgt_text, pred_text)
            edit_dist = calculate_edit_distance(tgt_text, pred_text)

            print(f"BLEU: {bleu:.3f}, CER: {cer:.3f}, Edit Dist: {edit_dist}")
            print("-" * 50)


In [32]:
def test_metrics_sanity_check():
    """Sanity check for evaluation metrics"""
    print("Testing evaluation metrics...")

    # Test BLEU
    ref = "yeh ek test sentence hai"
    hyp = "yeh ek test sentence hai"
    print(f"BLEU (identical): {calculate_bleu(ref, hyp):.4f}")

    hyp = "yeh test sentence hai"
    print(f"BLEU (missing word): {calculate_bleu(ref, hyp):.4f}")

    # Test CER
    ref = "hello world"
    hyp = "hello world"
    print(f"CER (identical): {calculate_cer(ref, hyp):.4f}")

    hyp = "helo wrold"
    print(f"CER (2 errors): {calculate_cer(ref, hyp):.4f}")

    # Test Edit Distance
    print(f"Edit distance (identical): {calculate_edit_distance('hello', 'hello')}")
    print(f"Edit distance (1 substitution): {calculate_edit_distance('hello', 'hallo')}")


In [33]:
test_metrics_sanity_check()

Testing evaluation metrics...
BLEU (identical): 1.0000
BLEU (missing word): 0.3611
CER (identical): 0.0000
CER (2 errors): 0.2727
Edit distance (identical): 0
Edit distance (1 substitution): 1


In [34]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


In [35]:
# Load data
src_sentences, tgt_sentences, urdu_vocab, roman_vocab = load_data_and_vocab()


In [36]:
# Create datasets
train_dataset, val_dataset, test_dataset = create_datasets(
    src_sentences, tgt_sentences, urdu_vocab, roman_vocab)


In [37]:
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)


In [38]:

# Initialize model
model = Seq2SeqModel().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding


In [39]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _حسن _وہی _ہے _حسن _جو _ظ الم
Target (Roman): husn_ va hi_ hai_ husn_ jo_ za li m_
Prediction: hua_ hua_ hua_ hua_ hua_ hua_ hua_ hua_ hua_
BLEU: 0.000, CER: 0.667, Edit Dist: 24
--------------------------------------------------
Source (Urdu): _مر ی _طرح _یوں _ہی _گ م _کر دہ _ر اہ _چھوڑ ے _گ ی
Target (Roman): miri_ ta ra h_ yu n hi_ gu m- ka r da - ra h_ chho de gi_
Prediction: is is is is is is is is is is is is is is is is is
BLEU: 0.000, CER: 0.702, Edit Dist: 40
--------------------------------------------------
Source (Urdu): _گ دا ئے _کو چ ۂ _مے _خ ان ہ _ن ا _مر اد _نہیں
Target (Roman): ga da-e- ku ch a -e- mai- kh an a_ na- mu ra d_ nahin_
Prediction: hua_ hua_ hua_ hua_ hua_ is is is is is is is is is is is is
BLEU: 0.000, CER: 0.815, Edit Dist: 44
--------------------------------------------------
Source (Urdu): _پھر _شوق _کر _رہا _ہے _خ ری دا ر _کی _ط لب
Target (Roman): phir_ shauq_ ka r_ ra ha _ hai_ kh ar i da r_ ki_ ta la b_
Predictio

In [40]:
print("Starting training...")

# Step 1: Train encoder only (freeze decoder)
print("\nStep 1: Training encoder (5 epochs)")
for param in model.decoder.parameters():
    param.requires_grad = False

optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.001)

train_model(model, train_loader, val_loader, criterion, optimizer, 10, device, roman_vocab)


Starting training...

Step 1: Training encoder (5 epochs)

Epoch 1/10
Train Loss: 6.0345, Perplexity: 417.5694, Accuracy: 0.0383
Val Loss:   6.0222, Perplexity: 412.4853, Accuracy: 0.0388, BLEU: 0.0020, CER: 0.8763, Edit Dist: 47.59

Epoch 2/10
Train Loss: 6.0159, Perplexity: 409.8865, Accuracy: 0.0383
Val Loss:   6.0137, Perplexity: 408.9999, Accuracy: 0.0387, BLEU: 0.0021, CER: 0.8593, Edit Dist: 46.81

Epoch 3/10
Train Loss: 6.0067, Perplexity: 406.1208, Accuracy: 0.0382
Val Loss:   6.0051, Perplexity: 405.4805, Accuracy: 0.0385, BLEU: 0.0021, CER: 0.8600, Edit Dist: 46.86

Epoch 4/10
Train Loss: 5.9995, Perplexity: 403.2369, Accuracy: 0.0380
Val Loss:   5.9998, Perplexity: 403.3501, Accuracy: 0.0386, BLEU: 0.0019, CER: 0.8512, Edit Dist: 46.42

Epoch 5/10
Train Loss: 5.9953, Perplexity: 401.5419, Accuracy: 0.0381
Val Loss:   5.9961, Perplexity: 401.8613, Accuracy: 0.0385, BLEU: 0.0019, CER: 0.8512, Edit Dist: 46.43

Epoch 6/10
Train Loss: 5.9921, Perplexity: 400.2733, Accuracy: 0.0

In [41]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _ہن س _دی ا _س ط ح _ذ ہ ن _عالم _پر
Target (Roman): ha n s_ diya_ sa t h -e- z eh n -e-a lam_ pa r_
Prediction: is is is is is is
BLEU: 0.000, CER: 0.830, Edit Dist: 39
--------------------------------------------------
Source (Urdu): _کی ا _ہو ں _تر ک _ن ر گ س _کا _تم ا شا
Target (Roman): kiya_ huun_ ta r k_ na r gi s_ ka_ tama sh a_
Prediction: i_ is is is is is is is is
BLEU: 0.000, CER: 0.711, Edit Dist: 32
--------------------------------------------------
Source (Urdu): _کچھ _نہ _تھا _ت یر ی _ق سم _تر ک _وفا _سے _پہلے
Target (Roman): kuchh_ na_ tha_ teri_ qa sam _ ta r k-e- vafa_ se_ pahle_
Prediction: is is is is is is is
BLEU: 0.000, CER: 0.842, Edit Dist: 48
--------------------------------------------------
Source (Urdu): _غیر _سمجھ ا _ہے _کہ _ل ذ ت _زخم _سو ز ن _میں _نہیں
Target (Roman): ghair_ samj ha _ hai_ ki_ la z za t_ zakh m-e- so za n_ men_ nahin_
Prediction: is is is is is is is
BLEU: 0.000, CER: 0.821, Edit Dist: 55
------------

In [42]:
# Step 2: Train decoder only (freeze encoder)
print("\nStep 2: Training decoder (10 epochs)")
for param in model.encoder.parameters():
    param.requires_grad = False
for param in model.decoder.parameters():
    param.requires_grad = True

optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.001)
train_model(model, train_loader, val_loader, criterion, optimizer, 15, device,roman_vocab)



Step 2: Training decoder (10 epochs)

Epoch 1/15
Train Loss: 4.5994, Perplexity: 99.4234, Accuracy: 0.1152
Val Loss:   4.3642, Perplexity: 78.5850, Accuracy: 0.1297, BLEU: 0.0239, CER: 0.5821, Edit Dist: 32.87

Epoch 2/15
Train Loss: 4.1827, Perplexity: 65.5446, Accuracy: 0.1415
Val Loss:   4.0057, Perplexity: 54.9106, Accuracy: 0.1551, BLEU: 0.0361, CER: 0.5212, Edit Dist: 29.36

Epoch 3/15
Train Loss: 3.8389, Perplexity: 46.4744, Accuracy: 0.1663
Val Loss:   3.6710, Perplexity: 39.2912, Accuracy: 0.1839, BLEU: 0.0517, CER: 0.4796, Edit Dist: 27.05

Epoch 4/15
Train Loss: 3.5218, Perplexity: 33.8454, Accuracy: 0.1909
Val Loss:   3.3436, Perplexity: 28.3198, Accuracy: 0.2111, BLEU: 0.0735, CER: 0.4336, Edit Dist: 24.49

Epoch 5/15
Train Loss: 3.2499, Perplexity: 25.7866, Accuracy: 0.2128
Val Loss:   3.0819, Perplexity: 21.7989, Accuracy: 0.2326, BLEU: 0.0935, CER: 0.4050, Edit Dist: 22.90

Epoch 6/15
Train Loss: 3.0184, Perplexity: 20.4587, Accuracy: 0.2325
Val Loss:   2.8858, Perplex

In [44]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _ہر _بر گ _گل _کے _پر د ے _میں _دل _بے _ق را ر _تھا
Target (Roman): ha r_ ba r g -e- gul_ ke_ pa r de _ men_ dil_ be- qa ra r_ tha_
Prediction: ha r_ ba r g gu gul_ ke_ pa de de _ men_ be- be- qa
BLEU: 0.335, CER: 0.317, Edit Dist: 20
--------------------------------------------------
Source (Urdu): _کوئی _جل د ی _میں _کوئی _دی ر _سے _جا نے _و ال ا
Target (Roman): koi_ jal di _ men_ koi_ de r_ se_ ja an e_ va a la _
Prediction: koi_ jal da-e- _ men_ koi_ di r_ se_ ja an e_ vaale_ a la
BLEU: 0.373, CER: 0.212, Edit Dist: 11
--------------------------------------------------
Source (Urdu): _چ اک _د ام اں _و _گ ری باں _کے _بھی _آ دا ب _ہیں _کچھ
Target (Roman): ch ak -e- da ma n -o- gi re ba n_ ke_ bhi_ a da b_ hain_ kuchh_
Prediction: ch ak d- da man_ n -o- gi re ba n_ ke_ bhi_ da b_ b_ kuchh_
BLEU: 0.478, CER: 0.159, Edit Dist: 10
--------------------------------------------------
Source (Urdu): _اور _اس _کی _چاہ ت _رکھ تے _ہیں _ہم _آج _ت لک _پھر _و 

In [46]:
#more 10 epochs
# Step 2: Train decoder only (freeze encoder)
print("\nStep 2: Training decoder (10 epochs)")
for param in model.encoder.parameters():
    param.requires_grad = False
for param in model.decoder.parameters():
    param.requires_grad = True

optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.001)
train_model(model, train_loader, val_loader, criterion, optimizer, 10, device,roman_vocab)



Step 2: Training decoder (10 epochs)

Epoch 1/10
Train Loss: 1.8988, Perplexity: 6.6779, Accuracy: 0.3539
Val Loss:   2.0659, Perplexity: 7.8928, Accuracy: 0.3503, BLEU: 0.3248, CER: 0.2554, Edit Dist: 14.48

Epoch 2/10
Train Loss: 1.8398, Perplexity: 6.2952, Accuracy: 0.3607
Val Loss:   2.0326, Perplexity: 7.6338, Accuracy: 0.3565, BLEU: 0.3331, CER: 0.2518, Edit Dist: 14.29

Epoch 3/10
Train Loss: 1.7983, Perplexity: 6.0397, Accuracy: 0.3677
Val Loss:   2.0389, Perplexity: 7.6818, Accuracy: 0.3555, BLEU: 0.3388, CER: 0.2464, Edit Dist: 13.96

Epoch 4/10
Train Loss: 1.7599, Perplexity: 5.8119, Accuracy: 0.3727
Val Loss:   2.0012, Perplexity: 7.3983, Accuracy: 0.3637, BLEU: 0.3531, CER: 0.2390, Edit Dist: 13.55

Epoch 5/10
Train Loss: 1.7172, Perplexity: 5.5690, Accuracy: 0.3792
Val Loss:   2.0020, Perplexity: 7.4040, Accuracy: 0.3627, BLEU: 0.3532, CER: 0.2421, Edit Dist: 13.73

Epoch 6/10
Train Loss: 1.6841, Perplexity: 5.3875, Accuracy: 0.3841
Val Loss:   1.9730, Perplexity: 7.1920

In [47]:
# Step 3: Train full model
print("\nStep 3: Training full model (10 epochs)")
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=0.0005)
train_model(model, train_loader, val_loader, criterion, optimizer, 20, device,roman_vocab)



Step 3: Training full model (10 epochs)

Epoch 1/20
Train Loss: 1.4942, Perplexity: 4.4559, Accuracy: 0.4103
Val Loss:   1.9587, Perplexity: 7.0904, Accuracy: 0.3773, BLEU: 0.3928, CER: 0.2224, Edit Dist: 12.61

Epoch 2/20
Train Loss: 1.4403, Perplexity: 4.2220, Accuracy: 0.4170
Val Loss:   1.9250, Perplexity: 6.8549, Accuracy: 0.3836, BLEU: 0.4018, CER: 0.2180, Edit Dist: 12.36

Epoch 3/20
Train Loss: 1.4077, Perplexity: 4.0866, Accuracy: 0.4246
Val Loss:   1.9336, Perplexity: 6.9146, Accuracy: 0.3835, BLEU: 0.4066, CER: 0.2142, Edit Dist: 12.14

Epoch 4/20
Train Loss: 1.3919, Perplexity: 4.0227, Accuracy: 0.4258
Val Loss:   1.9198, Perplexity: 6.8195, Accuracy: 0.3849, BLEU: 0.4040, CER: 0.2183, Edit Dist: 12.36

Epoch 5/20
Train Loss: 1.3660, Perplexity: 3.9197, Accuracy: 0.4285
Val Loss:   1.9646, Perplexity: 7.1319, Accuracy: 0.3840, BLEU: 0.4126, CER: 0.2147, Edit Dist: 12.15

Epoch 6/20
Train Loss: 1.3432, Perplexity: 3.8314, Accuracy: 0.4330
Val Loss:   1.9199, Perplexity: 6.8

In [48]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _ہے _بس ک ہ _ہر _اک _ا ن _کے _ا شا رے _میں _نش اں _اور
Target (Roman): hai_ ba s - ki_ ha r_ ik_ un_ ke_ is ha re_ men_ ni sh an _ aur_
Prediction: hai_ ba s ki_ ki_ ha r_ ik_ un_ ke_ ash re_ re_ men_ na sh an
BLEU: 0.372, CER: 0.250, Edit Dist: 16
--------------------------------------------------
Source (Urdu): _جل _گیا _اپنا _نش ی من _تو _کوئی _بات _نہیں
Target (Roman): jal _ gaya_ apna_ na sh e man_ to_ koi_ baat_ nahin_
Prediction: jal _ gaya_ vo_ na sh in_ in_ to_ koi_ nahin_
BLEU: 0.161, CER: 0.288, Edit Dist: 15
--------------------------------------------------
Source (Urdu): _اپنے _بس م ل _سے _یہ _کہ تا _تھا _د م _ن ز ع _وہ _شو خ
Target (Roman): apne_ bi s mil _ se_ ye_ ka h ta _ tha_ da m-e- naz a_ vo_ sh o kh _
Prediction: apne_ bi s mil _ se_ ye_ ka h ta _ tha_ dush na ma _ vo_ sh
BLEU: 0.580, CER: 0.235, Edit Dist: 16
--------------------------------------------------
Source (Urdu): _ہے _یہ _تک ی ہ _تری _ع ط ا ؤں _پر
Target (Roman): h

In [51]:
# Step 4: Fine-tune with low learning rate and decay
print("\nStep 4: Fine-tuning with learning rate decay (10 epochs)")
optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

for epoch in range(10):
    train_model(model, train_loader, val_loader, criterion, optimizer, 1, device, roman_vocab)
    scheduler.step()
    print(f"Learning rate: {scheduler.get_last_lr()[0]:.6f}")



Step 4: Fine-tuning with learning rate decay (10 epochs)

Epoch 1/1
Train Loss: 1.0495, Perplexity: 2.8562, Accuracy: 0.4730
Val Loss:   1.9892, Perplexity: 7.3097, Accuracy: 0.3983, BLEU: 0.4433, CER: 0.2000, Edit Dist: 11.30
Learning rate: 0.000090

Epoch 1/1
Train Loss: 1.0177, Perplexity: 2.7669, Accuracy: 0.4818
Val Loss:   1.9763, Perplexity: 7.2159, Accuracy: 0.3998, BLEU: 0.4444, CER: 0.1997, Edit Dist: 11.28
Learning rate: 0.000081

Epoch 1/1
Train Loss: 1.0060, Perplexity: 2.7348, Accuracy: 0.4838
Val Loss:   1.9791, Perplexity: 7.2362, Accuracy: 0.4010, BLEU: 0.4456, CER: 0.1982, Edit Dist: 11.20
Learning rate: 0.000073

Epoch 1/1
Train Loss: 0.9928, Perplexity: 2.6988, Accuracy: 0.4870
Val Loss:   1.9775, Perplexity: 7.2245, Accuracy: 0.4013, BLEU: 0.4480, CER: 0.1965, Edit Dist: 11.11
Learning rate: 0.000066

Epoch 1/1
Train Loss: 0.9820, Perplexity: 2.6698, Accuracy: 0.4892
Val Loss:   1.9798, Perplexity: 7.2415, Accuracy: 0.4019, BLEU: 0.4486, CER: 0.1975, Edit Dist: 11

In [52]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _وہی ں _پہ _تو ڑ ے _ہیں _ی ار وں _ن ے _آج _پی م ان ے
Target (Roman): va hi n_ pe_ to de _ hain_ ya ro n_ ne_ aaj_ pai ma ne_
Prediction: va hi n_ pe_ to d de hain_ ya ro n_ ne_ aaj_ pi ya n
BLEU: 0.542, CER: 0.127, Edit Dist: 7
--------------------------------------------------
Source (Urdu): _آئے _ہو _وقت _صبح _رہے _ر ات _بھر _کہاں
Target (Roman): aae_ ho_ vaq t-e- subh_ ra he_ raat_ bha r_ ka ha n_
Prediction: aae_ ho_ vaqt_ subh_ ra he_ raat_ bha r_ ka
BLEU: 0.524, CER: 0.192, Edit Dist: 10
--------------------------------------------------
Source (Urdu): _ہم _بھی _اب _گھر _سے _کم _ن ک ل تے _ہیں
Target (Roman): ham_ bhi_ ab_ gha r_ se_ kam_ nik a lte_ hain_
Prediction: ham_ bhi_ ab_ gha r_ se_ kam_ nik a lte_ hain_
BLEU: 1.000, CER: 0.000, Edit Dist: 0
--------------------------------------------------
Source (Urdu): _ز اہ د _ن ے _کچھ _اس _ا ند ا ز _سے _پی _سا قی _کی _نگاہ یں _پڑ نے _لگ یں
Target (Roman): za hi d_ ne_ kuchh_ is _ anda z_ se_ pi 

In [53]:
# Evaluation
print("\nEvaluation Results:")
evaluate_model(model, test_loader, roman_vocab, device)



Evaluation Results:
Test Loss: 2.0089, Perplexity: 7.4549
BLEU Score: 0.4518
Character Error Rate: 0.1949
Average Edit Distance: 10.96


(0.45178210247696443, 0.19485231869574338, 10.957174816235218)

In [55]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _ہیں _دل ی لی ں _ترے _خ لا ف _مگر
Target (Roman): hain_ da li le n_ tire_ kh il af _ magar_
Prediction: hain_ dil_ in li n_ tire_ kh a af _ magar_
BLEU: 0.210, CER: 0.195, Edit Dist: 8
--------------------------------------------------
Source (Urdu): _بد ن _کو _بے _لب اد ہ _کر _ل یا _کی ا
Target (Roman): bad an _ ko_ be- li bad a_ ka r_ liya_ kya_
Prediction: bad an _ ko_ be- la j de _ ka r_ liya_ kya_
BLEU: 0.484, CER: 0.116, Edit Dist: 5
--------------------------------------------------
Source (Urdu): _درد _اپنا تا _ہے _پر ائے _کون
Target (Roman): dard_ ap na ta _ hai_ pa ra e_ kaun_
Prediction: dard_ apna_ ta ta _ hai_ pa r
BLEU: 0.285, CER: 0.389, Edit Dist: 14
--------------------------------------------------
Source (Urdu): _زم ان ہ _کو د _پڑ ا _آگ _میں _یہی _کہہ _کر
Target (Roman): zamana_ ku u d_ pa da _ aa g_ men_ ya hi_ ka h_ ka r_
Prediction: zamana_ ko_ ka ko_ ko_ ka ka aa men_ ya hi_ ka h_
BLEU: 0.258, CER: 0.396, Edit Dist: 21
------

In [56]:
save_path = "/content/drive/MyDrive/Model2/urdu_roman_nmt_model.pth"
# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'urdu_vocab': urdu_vocab,
    'roman_vocab': roman_vocab
}, save_path)
print("Model saved as 'urdu_roman_nmt_model.pth'")



Model saved as 'urdu_roman_nmt_model.pth'
